In [1]:
from analysis import load_metrics
from datetime import datetime
import pandas as pd

## Aggregating Meditation (value type is none)


In [2]:
meditation = load_metrics(1, "Meditation", "US/Pacific")
meditation

,recorded_at
1,2026-01-06 10:03:48-08:00
2,2026-01-07 08:32:24-08:00
3,2026-01-08 19:24:25-08:00
4,2026-01-11 07:51:57-08:00
5,2026-01-13 07:42:20-08:00
6,2026-01-16 09:58:48-08:00
7,2026-01-17 10:14:07-08:00
8,2026-01-19 15:11:58-08:00
9,2026-01-21 10:00:46-08:00
10,2026-01-23 14:10:39-08:00


Lets make sure that the datetimes returned are actually converted to a new timezone, i.e., the timestamp is kept as-is, but the timezones are changed, instead of localized or attached, where the timestamp is changed to keep the datetime as-is.


In [3]:
load_metrics(1, "Meditation", "Asia/Kolkata")

,recorded_at
1,2026-01-06 23:33:48+05:30
2,2026-01-07 22:02:24+05:30
3,2026-01-09 08:54:25+05:30
4,2026-01-11 21:21:57+05:30
5,2026-01-13 21:12:20+05:30
6,2026-01-16 23:28:48+05:30
7,2026-01-17 23:44:07+05:30
8,2026-01-20 04:41:58+05:30
9,2026-01-21 23:30:46+05:30
10,2026-01-24 03:40:39+05:30


In [4]:
load_metrics(1, "Meditation", "UTC")

,recorded_at
1,2026-01-06 18:03:48+00:00
2,2026-01-07 16:32:24+00:00
3,2026-01-09 03:24:25+00:00
4,2026-01-11 15:51:57+00:00
5,2026-01-13 15:42:20+00:00
6,2026-01-16 17:58:48+00:00
7,2026-01-17 18:14:07+00:00
8,2026-01-19 23:11:58+00:00
9,2026-01-21 18:00:46+00:00
10,2026-01-23 22:10:39+00:00


In [5]:
meditation.info()

<class 'pandas.DataFrame'>
Index: 16 entries, 1 to 16
Data columns (total 1 columns):
 #   Column       Non-Null Count  Dtype                    
---  ------       --------------  -----                    
 0   recorded_at  16 non-null     datetime64[s, US/Pacific]
dtypes: datetime64[s, US/Pacific](1)
memory usage: 256.0 bytes


### Aggregating by Timeseries

Grouping by timestamps is done via the `.resample()` method. It downsamples by-second data into daily/weekly/monthly/etc. aggregates. It is useful to keep the split-apply-combine paradigm in mind. The splitting is done by timestamps, once the data is split into groups, I need to apply some function to the group. By default the function is applied to all the columns in the group. I can specify that the function should only be applied to numeric columns, etc., but the usual pattern I have seen is that you usually first select the columns you want to apply the function to, then split the projected dataframe and apply the function.


In [6]:
# Split: On the weekly recorded_at field
# Apply: The size function which will count the number of columns in the dataframes. Even though
# apart from recorded_on, there are no other columns, it is ok. The count function will not work
# because it needs other columns to count the non-missing values in them.
# The result is a series because there no other columns.
medgrps = meditation.resample("W", on="recorded_at").size()
print(type(medgrps))
medgrps

<class 'pandas.Series'>


recorded_at
2026-01-11 00:00:00-08:00    4
2026-01-18 00:00:00-08:00    3
2026-01-25 00:00:00-08:00    4
2026-02-01 00:00:00-08:00    4
2026-02-08 00:00:00-08:00    1
Freq: W-SUN, dtype: int64

In [7]:
# Here is what count will give me
meditation.resample("W", on="recorded_at").count()

""
recorded_at
2026-01-11 00:00:00-08:00
2026-01-18 00:00:00-08:00
2026-01-25 00:00:00-08:00
2026-02-01 00:00:00-08:00
2026-02-08 00:00:00-08:00


In [8]:
# Iterating through the series will give 2-tuple, the index value and the data value
for idx, sz in medgrps.items():
    print(idx, sz)

2026-01-11 00:00:00-08:00 4
2026-01-18 00:00:00-08:00 3
2026-01-25 00:00:00-08:00 4
2026-02-01 00:00:00-08:00 4
2026-02-08 00:00:00-08:00 1


In [9]:
# I can just get the index
medgrps.index

DatetimeIndex(['2026-01-11 00:00:00-08:00', '2026-01-18 00:00:00-08:00',
               '2026-01-25 00:00:00-08:00', '2026-02-01 00:00:00-08:00',
               '2026-02-08 00:00:00-08:00'],
              dtype='datetime64[s, US/Pacific]', name='recorded_at', freq='W-SUN')

## Aggregating Weights (numeric value)


In [10]:
weights = load_metrics(1, "Weight", "US/Pacific")
weights

,recorded_at,value
17,2026-01-06 07:41:18-08:00,184.6
18,2026-01-07 12:32:06-08:00,184.3
19,2026-01-08 16:59:25-08:00,184.6
20,2026-01-09 06:47:27-08:00,184.2
21,2026-01-10 15:45:42-08:00,184.3
22,2026-01-11 12:53:27-08:00,183.7
23,2026-01-12 11:20:13-08:00,183.0
24,2026-01-13 17:15:35-08:00,183.3
25,2026-01-14 12:35:00-08:00,183.6
26,2026-01-15 18:49:34-08:00,183.6


In [11]:
# Unlike meditation groups, this one is a dataframe because there was at least one additional column
# apart from recorded_at
wtgrps = weights.resample("W", on="recorded_at").mean()
print(type(wtgrps))
wtgrps

<class 'pandas.DataFrame'>


,value
recorded_at,
2026-01-11 00:00:00-08:00,184.283333
2026-01-18 00:00:00-08:00,183.085714
2026-01-25 00:00:00-08:00,181.571429
2026-02-01 00:00:00-08:00,181.485714
2026-02-08 00:00:00-08:00,181.400000


In [12]:
# When I iterate through any dataframe using items(), it will yeild each column
# as a series. The column will consist of a 2-tuple, the name of the column and
# the series. In this case there is only a single column named value.
for colname, series in wtgrps.items():
    print(f"*** {colname} ****")

    # Now I can iterate through the series as usual
    for idx, avg in series.items():
        print(idx, avg)

*** value ****
2026-01-11 00:00:00-08:00 184.28333333333333
2026-01-18 00:00:00-08:00 183.08571428571426
2026-01-25 00:00:00-08:00 181.57142857142858
2026-02-01 00:00:00-08:00 181.4857142857143
2026-02-08 00:00:00-08:00 181.4


In [13]:
# There is a second way I can iterate through the dataframe and that is using
# iterrows, though the API is a bit funky. The iterator will yeild a 2-tuple, the
# first element is the index value, and the second element is a series. The series
# will contain each column name as the index, and the column value as the row value.
# So this series will have as many rows as the number of columns in the original df.
# In this case there is only a single column, so the series will contain a single row
# with the index as the colname and the row value as the value of that column.
for idx, val in wtgrps.iterrows():
    print(idx)
    for validx, valval in val.items():
        print(validx, valval)
    print("---")

2026-01-11 00:00:00-08:00
value 184.28333333333333
---
2026-01-18 00:00:00-08:00
value 183.08571428571426
---
2026-01-25 00:00:00-08:00
value 181.57142857142858
---
2026-02-01 00:00:00-08:00
value 181.4857142857143
---
2026-02-08 00:00:00-08:00
value 181.4
---


## Aggregating Mood (categorical value)


In [14]:
mood = load_metrics(1, "Mood", "US/Pacific")
mood.head(15)

,recorded_at,value
46,2026-01-06 07:12:17-08:00,sad
45,2026-01-06 13:18:10-08:00,serene
47,2026-01-06 16:19:30-08:00,serene
49,2026-01-07 13:12:08-08:00,angry
48,2026-01-07 17:55:54-08:00,angry
51,2026-01-08 10:29:22-08:00,serene
50,2026-01-08 10:49:19-08:00,sad
54,2026-01-09 08:31:05-08:00,angry
53,2026-01-09 12:35:23-08:00,sad
52,2026-01-09 16:37:22-08:00,angry


In [15]:
# Again, count will not work because I am consuming both the columns - recorded_at and value - as
# group keys. After that there are no more columns left to count.
moodgrps = mood.groupby([pd.Grouper(key="recorded_at", freq="W"), "value"]).size()
print(type(moodgrps))
moodgrps

<class 'pandas.Series'>


recorded_at                value 
2026-01-11 00:00:00-08:00  happy     2
                           sad       4
                           serene    3
                           angry     5
2026-01-18 00:00:00-08:00  happy     6
                           sad       3
                           serene    6
                           angry     2
2026-01-25 00:00:00-08:00  happy     4
                           sad       4
                           serene    3
                           angry     4
2026-02-01 00:00:00-08:00  happy     4
                           sad       3
                           serene    1
                           angry     7
2026-02-08 00:00:00-08:00  sad       1
                           angry     1
dtype: int64

In [16]:
# I can "pivot" the resulting series, but I don't expect to use this.
mood.groupby([pd.Grouper(key="recorded_at", freq="W"), "value"]).size().unstack(
    fill_value=0
)

value,happy,sad,serene,angry
recorded_at,,,,
2026-01-11 00:00:00-08:00,2,4,3,5
2026-01-18 00:00:00-08:00,6,3,6,2
2026-01-25 00:00:00-08:00,4,4,3,4
2026-02-01 00:00:00-08:00,4,3,1,7
2026-02-08 00:00:00-08:00,0,1,0,1


In [17]:
# Here the index is a Multi-Index, i.e., it is composed of multiple elements
# in this case two, the timeseries key, and the value key.
for index, sz in moodgrps.items():
    print(index, sz)

(Timestamp('2026-01-11 00:00:00-0800', tz='US/Pacific'), 'happy') 2
(Timestamp('2026-01-11 00:00:00-0800', tz='US/Pacific'), 'sad') 4
(Timestamp('2026-01-11 00:00:00-0800', tz='US/Pacific'), 'serene') 3
(Timestamp('2026-01-11 00:00:00-0800', tz='US/Pacific'), 'angry') 5
(Timestamp('2026-01-18 00:00:00-0800', tz='US/Pacific'), 'happy') 6
(Timestamp('2026-01-18 00:00:00-0800', tz='US/Pacific'), 'sad') 3
(Timestamp('2026-01-18 00:00:00-0800', tz='US/Pacific'), 'serene') 6
(Timestamp('2026-01-18 00:00:00-0800', tz='US/Pacific'), 'angry') 2
(Timestamp('2026-01-25 00:00:00-0800', tz='US/Pacific'), 'happy') 4
(Timestamp('2026-01-25 00:00:00-0800', tz='US/Pacific'), 'sad') 4
(Timestamp('2026-01-25 00:00:00-0800', tz='US/Pacific'), 'serene') 3
(Timestamp('2026-01-25 00:00:00-0800', tz='US/Pacific'), 'angry') 4
(Timestamp('2026-02-01 00:00:00-0800', tz='US/Pacific'), 'happy') 4
(Timestamp('2026-02-01 00:00:00-0800', tz='US/Pacific'), 'sad') 3
(Timestamp('2026-02-01 00:00:00-0800', tz='US/Pacific

In [18]:
# I can index with the first index to get all the elements that match it.
tp = moodgrps[pd.Timestamp("2026-01-11 00:00:00-0800", tz="US/Pacific")]
print(type(tp))
print(tp)

<class 'pandas.Series'>
value
happy     2
sad       4
serene    3
angry     5
dtype: int64


In [19]:
# Just get the first index
moodgrps.index.get_level_values(0).unique()

DatetimeIndex(['2026-01-11 00:00:00-08:00', '2026-01-18 00:00:00-08:00',
               '2026-01-25 00:00:00-08:00', '2026-02-01 00:00:00-08:00',
               '2026-02-08 00:00:00-08:00'],
              dtype='datetime64[s, US/Pacific]', name='recorded_at', freq=None)

In [20]:
# This does not seem very useful
moodgrps.index.get_level_values(1)

CategoricalIndex(['happy', 'sad', 'serene', 'angry', 'happy', 'sad', 'serene',
                  'angry', 'happy', 'sad', 'serene', 'angry', 'happy', 'sad',
                  'serene', 'angry', 'sad', 'angry'],
                 categories=['happy', 'sad', 'serene', 'angry'], ordered=False, dtype='category', name='value')

In [21]:
for week in moodgrps.index.get_level_values(0).unique():
    print(f"In the week ending {week}:")
    for moodcat, count in moodgrps[week].items():
        print(f"\tI was {moodcat} {count} number of times.")

In the week ending 2026-01-11 00:00:00-08:00:
	I was happy 2 number of times.
	I was sad 4 number of times.
	I was serene 3 number of times.
	I was angry 5 number of times.
In the week ending 2026-01-18 00:00:00-08:00:
	I was happy 6 number of times.
	I was sad 3 number of times.
	I was serene 6 number of times.
	I was angry 2 number of times.
In the week ending 2026-01-25 00:00:00-08:00:
	I was happy 4 number of times.
	I was sad 4 number of times.
	I was serene 3 number of times.
	I was angry 4 number of times.
In the week ending 2026-02-01 00:00:00-08:00:
	I was happy 4 number of times.
	I was sad 3 number of times.
	I was serene 1 number of times.
	I was angry 7 number of times.
In the week ending 2026-02-08 00:00:00-08:00:
	I was sad 1 number of times.
	I was angry 1 number of times.


## Aggregating Meal (categorical properties but no value)


In [27]:
meal = load_metrics(1, "Meal", "US/Pacific")
meal.head(20)

,recorded_at,source,taste,is_filling,healthy
108,2026-01-06 08:08:31-08:00,home-cooked,edible,True,very
109,2026-01-06 13:22:15-08:00,tiffin,delicious,True,very
110,2026-01-06 19:05:40-08:00,take-out,edible,False,very
111,2026-01-07 08:29:30-08:00,take-out,delicious,True,medium
112,2026-01-07 13:00:04-08:00,take-out,edible,False,medium
113,2026-01-07 19:14:26-08:00,tiffin,bad,False,very
114,2026-01-08 08:06:28-08:00,take-out,delicious,True,no
115,2026-01-08 13:25:06-08:00,tiffin,delicious,False,very
116,2026-01-08 19:01:42-08:00,tiffin,edible,True,very
117,2026-01-09 08:17:20-08:00,home-cooked,delicious,True,no


In [26]:
mealgrps = meal.groupby([pd.Grouper(key="recorded_at", freq="W"), "healthy"]).size()
mealgrps

recorded_at                healthy
2026-01-11 00:00:00-08:00  very       9
                           medium     4
                           no         5
2026-01-18 00:00:00-08:00  very       9
                           medium     7
                           no         5
2026-01-25 00:00:00-08:00  very       6
                           medium     8
                           no         7
2026-02-01 00:00:00-08:00  very       6
                           medium     8
                           no         7
2026-02-08 00:00:00-08:00  very       3
dtype: int64

In [ ]:
# How healthy were home cooked delicious meals?
meal[(meal["source"] == "home-cooked") & (meal["taste"] == "delicious")].groupby(
    [pd.Grouper(key="recorded_at", freq="W"), "healthy"]
).size()

recorded_at                healthy
2026-01-11 00:00:00-08:00  no         3
2026-01-18 00:00:00-08:00  very       1
                           medium     2
2026-01-25 00:00:00-08:00  very       1
dtype: int64

## Aggregating Blood Glucose (categorical properties and numeric value)


In [32]:
glucose = load_metrics(1, "Blood Glucose", "US/Pacific")
glucose.head(10)

,recorded_at,value,event,delta
192,2026-01-06 06:46:04-08:00,104.0,fasting,NaN
193,2026-01-06 10:08:36-08:00,170.1,breakfast,two-hours-after
194,2026-01-07 06:07:06-08:00,110.0,fasting,NaN
195,2026-01-07 09:26:39-08:00,177.4,breakfast,one-hour-after
196,2026-01-07 16:08:28-08:00,128.0,workout,one-hour-after
197,2026-01-08 07:24:49-08:00,88.8,fasting,NaN
198,2026-01-08 09:19:15-08:00,176.9,breakfast,one-hour-after
199,2026-01-09 06:50:21-08:00,89.2,fasting,NaN
200,2026-01-10 07:51:51-08:00,89.1,fasting,NaN
201,2026-01-10 09:10:38-08:00,154.5,breakfast,one-hour-after


In [34]:
glucose[["recorded_at", "value", "delta"]].groupby(
    [pd.Grouper(key="recorded_at", freq="W"), "delta"]
).mean()

value
recorded_at               delta                      
2026-01-11 00:00:00-08:00 one-hour-after   159.720000
                          two-hours-after  170.100000
2026-01-18 00:00:00-08:00 two-hours-after  146.350000
2026-01-25 00:00:00-08:00 one-hour-after   128.700000
                          two-hours-after  145.600000
2026-02-01 00:00:00-08:00 one-hour-after   122.233333
                          two-hours-after  149.133333
2026-02-08 00:00:00-08:00 one-hour-after   111.800000

In [ ]:
# How are my sugar levels around breakfast?
glucose.loc[glucose["event"] == "breakfast", ["recorded_at", "value", "delta"]].groupby(
    [pd.Grouper(key="recorded_at", freq="W"), "delta"]
).mean()

value
recorded_at               delta                  
2026-01-11 00:00:00-08:00 one-hour-after   167.65
                          two-hours-after  170.10
2026-01-18 00:00:00-08:00 two-hours-after  146.35
2026-01-25 00:00:00-08:00 one-hour-after   145.20
                          two-hours-after  162.10
2026-02-01 00:00:00-08:00 one-hour-after   124.50
                          two-hours-after  166.05

## Aggregate Hike (mixed properties, numeric value)


In [42]:
hike = load_metrics(1, "Hike", "US/Pacific")
hike

,recorded_at,value,loop_length,elevation_gain,landscape
243,2026-01-07 19:38:03-08:00,49.0,2.7,174.0,ridge
244,2026-01-12 11:32:08-08:00,220.0,11.2,379.0,woods
245,2026-01-15 13:51:53-08:00,155.0,7.4,378.0,woods
246,2026-01-17 18:16:30-08:00,83.0,5.2,238.0,ridge
247,2026-01-26 19:46:02-08:00,134.0,8.7,252.0,lake
248,2026-01-31 13:56:06-08:00,160.0,6.4,259.0,woods
249,2026-02-01 13:26:01-08:00,199.0,10.1,777.0,river


In [43]:
hike["elevation_gain_bin"] = pd.cut(
    hike["elevation_gain"], bins=3, labels=["low", "medium", "high"]
)
hike

,recorded_at,value,loop_length,elevation_gain,landscape,elevation_gain_bin
243,2026-01-07 19:38:03-08:00,49.0,2.7,174.0,ridge,low
244,2026-01-12 11:32:08-08:00,220.0,11.2,379.0,woods,medium
245,2026-01-15 13:51:53-08:00,155.0,7.4,378.0,woods,medium
246,2026-01-17 18:16:30-08:00,83.0,5.2,238.0,ridge,low
247,2026-01-26 19:46:02-08:00,134.0,8.7,252.0,lake,low
248,2026-01-31 13:56:06-08:00,160.0,6.4,259.0,woods,low
249,2026-02-01 13:26:01-08:00,199.0,10.1,777.0,river,high


In [44]:
hike[["recorded_at", "value", "elevation_gain_bin"]].groupby(
    [pd.Grouper(key="recorded_at", freq="W"), "elevation_gain_bin"]
).mean()

value
recorded_at               elevation_gain_bin       
2026-01-11 00:00:00-08:00 low                  49.0
2026-01-18 00:00:00-08:00 low                  83.0
                          medium              187.5
2026-02-01 00:00:00-08:00 low                 147.0
                          high                199.0